In [29]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
import os

endpoint = os.environ["VISION_ENDPOINT"]
key = os.environ["VISION_KEY"]


client = ImageAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

In [30]:
with open("/Users/gims43077/Downloads/LIT.png", "rb") as f:
    image_data = f.read()

visual_features = [
    VisualFeatures.CAPTION,          # 이미지 한 줄 설명
    VisualFeatures.DENSE_CAPTIONS,   # 이미지 여러 영역별 설명
    VisualFeatures.TAGS,             # 이미지 태그
    VisualFeatures.OBJECTS,          # 객체 위치
    VisualFeatures.PEOPLE,           # 사람 위치
    VisualFeatures.READ,             # 이미지 속 글자 OCR
    VisualFeatures.SMART_CROPS,      # 썸네일 crop 정보
]


In [31]:
try:
    result = client.analyze(
        image_data=image_data,
        visual_features=visual_features,
        smart_crops_aspect_ratios=[0.9, 1.33],
        language="en"
    )

    print("분석 성공")

    print("========== Azure AI Vision Image Analysis Result ==========")

    # 1. 이미지 전체 설명
    if result.caption is not None:
        print("\n[1] 이미지 설명 Caption")
        print(f"- {result.caption.text}")
        print(f"- confidence: {result.caption.confidence:.4f}")

    # 2. 영역별 설명
    if result.dense_captions is not None:
        print("\n[2] 영역별 설명 Dense Captions")
        for i, caption in enumerate(result.dense_captions.list, start=1):
            print(f"{i}. {caption.text}")
            print(f"   box: {caption.bounding_box}")
            print(f"   confidence: {caption.confidence:.4f}")

    # 3. 태그
    if result.tags is not None:
        print("\n[3] 태그 Tags")
        for tag in result.tags.list:
            print(f"- {tag.name}: {tag.confidence:.4f}")

    # 4. 객체 위치
    if result.objects is not None:
        print("\n[4] 객체 위치 Objects")
        for obj in result.objects.list:
            tag = obj.tags[0]
            print(f"- {tag.name}: {tag.confidence:.4f}")
            print(f"  box: {obj.bounding_box}")

    # 5. 사람 위치
    if result.people is not None:
        print("\n[5] 사람 위치 People")
        for person in result.people.list:
            print(f"- confidence: {person.confidence:.4f}")
            print(f"  box: {person.bounding_box}")

    # 6. 이미지 속 글자 OCR
    if result.read is not None:
        print("\n[6] 이미지 속 글자 Read OCR")
        for block in result.read.blocks:
            for line in block.lines:
                print(f"- line: {line.text}")
                print(f"  polygon: {line.bounding_polygon}")

    # 7. 썸네일 crop 정보
    if result.smart_crops is not None:
        print("\n[7] 썸네일용 Smart Crops")
        for crop in result.smart_crops.list:
            print(f"- aspect ratio: {crop.aspect_ratio}")
            print(f"  box: {crop.bounding_box}")

    # 8. 메타데이터
    print("\n[8] 이미지 메타데이터")
    print(f"- width: {result.metadata.width}")
    print(f"- height: {result.metadata.height}")
    print(f"- model version: {result.model_version}")

except HttpResponseError as e:
    print("Azure AI Vision 호출 실패")
    print("Status code:", e.status_code)
    print("Reason:", e.reason)
    print("Message:", e.error.message if e.error else e.message)

분석 성공
========== Azure AI Vision Image Analysis Result ==========

[1] 이미지 설명 Caption
- a screen shot of a phone
- confidence: 0.7704

[2] 영역별 설명 Dense Captions
1. a screen shot of a phone
   box: {'x': 0, 'y': 0, 'w': 1587, 'h': 2245}
   confidence: 0.7704
2. a black and white sign with black text
   box: {'x': 295, 'y': 166, 'w': 971, 'h': 321}
   confidence: 0.6688
3. a logo with a blue and green gradient
   box: {'x': 1176, 'y': 1318, 'w': 103, 'h': 108}
   confidence: 0.6168
4. a screenshot of a computer
   box: {'x': 40, 'y': 1038, 'w': 1454, 'h': 992}
   confidence: 0.7234
5. a close up of a card
   box: {'x': 98, 'y': 114, 'w': 1434, 'h': 944}
   confidence: 0.7155
6. a screenshot of a phone
   box: {'x': 3, 'y': 0, 'w': 1535, 'h': 2158}
   confidence: 0.8006
7. a qr code on a white background
   box: {'x': 1201, 'y': 742, 'w': 264, 'h': 265}
   confidence: 0.8654
8. a black and white image of a paperclip
   box: {'x': 1157, 'y': 1306, 'w': 121, 'h': 498}
   confidence: 0.7001
